In [1]:
import os
import re
import polars as pl
from tqdm import tqdm

from typing import List, Dict, Any

from llm_benchmark.utils.enums import QuestionType
from llm_benchmark.data.eval import get_ids_from_row, get_actual_row
from llm_benchmark.utils.dataset import Dataset, DatasetModule
from llm_benchmark import config as cfg

import traceback

from llm_benchmark.utils.benchmark import seshat_setup

dataset: Dataset = seshat_setup(
    seshat_cache_dir="/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/seshat", 
    force=False
    )

def parse_output(output: str) -> tuple[str, str]:
    output = output or ""

    reasoning_match = re.search(
        r"REASONING:\s*(.*?)(?=\s*ANSWER:|$)",
        output,
        flags=re.IGNORECASE | re.DOTALL,
    )
    answer_match = re.search(
        r"ANSWER:\s*['\"`\s]*([A-E])['\"`\s.,;:]*",
        output,
        flags=re.IGNORECASE,
    )

    reasoning = reasoning_match.group(1).strip() if reasoning_match else ""
    answer = answer_match.group(1).upper() if answer_match else "failed"

    return reasoning, answer

def parse_csv_to_df(dataset: Dataset, model_name: str, model_database_source: str, category: str,
                    path_to_types: str, question_type: QuestionType,) -> List[pl.DataFrame]:
    try:
        csv_paths: List[str] = os.listdir(os.path.join(path_to_types, question_type.name.lower()))
    except:
        return []
    dfs: List[pl.DataFrame] = []
    
    for csv_path in tqdm(csv_paths, desc="     Loading csvs"):
        csv: pl.DataFrame = pl.read_csv(os.path.join(path_to_types, question_type.name.lower(), csv_path))
        new_dicts: List[Dict[str, Any]] = []

        endpoint_identifier: str = csv[1, 0].replace(cfg.ENDPOINT_URL, "")[:-1]
        dataset_module: DatasetModule = dataset.get_module(identifier=endpoint_identifier.replace(cfg.ENDPOINT_URL, "",))
        dataset_df: pl.DataFrame = dataset_module.get_entries()

        for row in csv.iter_rows(named=True):
            reasoning, answer = parse_output(row["output"])
            entry_idx: int = int(row["entry_idx"])
            row_actual: Dict[str, Any] = get_actual_row(dataset_df=dataset_df, entry_idx=entry_idx)

            row_actual = get_actual_row(dataset_df=dataset_df, entry_idx=entry_idx)
            actual_answer = row_actual.get("polity_validity")
            if actual_answer is None:
                continue

            actual_answer = str(actual_answer).strip().lower().rstrip(".")

            option_map = {
                letter.upper(): value.lower()
                for letter, value in re.findall(
                    r"\b([A-E])\s*=\s*(present|absent|unknown)\b",
                    row["message"] or "",
                    flags=re.IGNORECASE,
                )
            }

            predicted_answer = option_map.get(answer.upper(), "")
            result = int(predicted_answer == actual_answer)

            try:
                ids: Dict[str, Any] = get_ids_from_row(dataset.grouping, row_actual, endpoint_identifier)
            except Exception as e:
                print(f"ID Error: {type(e).__name__}: {e}")
                print(f"Endpoint: {endpoint_identifier}")
                print(f"Row keys: {list(row_actual.keys())}")
                print(traceback.format_exc())
                ids: Dict[str, Any] = {}

            entry: Dict[str, Any] = {
                "endpoint_identifier": row["endpoint_identifier"],
                "entry_idx": entry_idx,
                "model": row["model"],
                "db_model": model_database_source,
                "reasoning": reasoning,

                "raw_model_answer": answer,
                "raw_ground_truth": actual_answer,
                "mapped_model_answer": predicted_answer,
                "result": result,
                "message": row["message"],
            }
            entry.update(ids)
            new_dicts.append(entry)
            
        dfs.append(pl.DataFrame(new_dicts))
    return dfs
        
            

def parse_all(path: str, question_type: QuestionType, dataset: Dataset) -> pl.DataFrame:
    df: pl.DataFrame = pl.DataFrame({})
    for company in os.listdir(path):
        model_path: str = os.path.join(path, company)
        models_per_db: List[str] = os.listdir(model_path) 
        for model in tqdm(models_per_db, desc=f"Loading model: {company}\n"):
            for model_version in os.listdir(os.path.join(model_path, model)):
                model_per_db_path: str = os.path.join(model_path, model, model_version)
                itms: List[str] = model_version.split('+')

                assert ('+' in model_version)

                model_name: str = itms[0]
                model_database_src: str = itms[1]

                for category in os.listdir(model_per_db_path):
                    path_to_types: str = os.path.join(model_per_db_path, category)
                    if category == ".DS_Store":
                        continue
                    
                    cat_dfs: List[pl.DataFrame] = parse_csv_to_df(
                        dataset=dataset,
                        model_name=model_name,
                        model_database_source=model_database_src,
                        category=category,
                        question_type=question_type,
                        path_to_types=path_to_types
                    ) 

                    try:
                        df = pl.concat([df, *cat_dfs], how="diagonal_relaxed")
                    except:
                        print("Failed to concat df")
    
    return df



/Users/apple/miniconda3/envs/llm_benchmark/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Attempting dataset refresh. False
core/macro-regions https://seshat-db.com/api/core/macro-regions/
Ignoring polity core/macro-regions as per configuration.
core/regions https://seshat-db.com/api/core/regions/
Ignoring polity core/regions as per configuration.
core/ngas https://seshat-db.com/api/core/ngas/
Ignoring polity core/ngas as per configuration.
core/polities https://seshat-db.com/api/core/polities/
Ignoring polity core/polities as per configuration.
core/capitals https://seshat-db.com/api/core/capitals/
Ignoring polity core/capitals as per configuration.
core/nga-polity-relations https://seshat-db.com/api/core/nga-polity-relations/
Ignoring polity core/nga-polity-relations as per configuration.
core/sections https://seshat-db.com/api/core/sections/
Ignoring polity core/sections as per configuration.
core/subsections https://seshat-db.com/api/core/subsections/
Ignoring polity core/subsections as per configuration.
core/variable-hierarchies https://seshat-db.com/api/core/variable

In [2]:

eval_src: str = "/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/eval/final"

df: pl.DataFrame = parse_all(
    path=eval_src,
    question_type=QuestionType.MULTIPLE_CHOICE,
    dataset=dataset
    )

Loading model: gemini
     Loading csvs: 0it [00:00, ?it/s]s]
     Loading csvs: 100%|██████████| 17/17 [00:01<00:00, 12.86it/s]
Loading model: gemini
     Loading csvs: 100%|██████████| 28/28 [00:01<00:00, 20.53it/s] 
Loading model: gemini
     Loading csvs: 100%|██████████| 28/28 [00:01<00:00, 18.82it/s] 
Loading model: gemini
Loading model: gemini4 [01:01<00:00, 16.77s/it]
: 100%|██████████| 4/4 [01:01<00:00, 15.47s/it]
Loading model: gpt
     Loading csvs: 100%|██████████| 20/20 [00:03<00:00,  5.96it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 100%|██████████| 20/20 [00:03<00:00,  5.92it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 0it [00:00, ?it/s]
Loading model: gpt
     Loading csvs: 100%|██████████| 43/43 [00:17<00:00,  2.43it/s]
Loading model: gpt
     Loading csvs: 0it [00:00, ?it/s]26.89s/it]
     Loading csvs: 100%|██████████| 49/

In [3]:
def group(df: pl.DataFrame, metric: str) -> pl.DataFrame:
    return (
        df
        .select(["model", "endpoint_identifier", "result", metric])
        .group_by(["model", "endpoint_identifier", metric])
        .agg(
            pl.col("result").mean().alias("per_metric")
        )
        .group_by(["model", metric])
        .agg(
            pl.format(
                "{} [{}, {}]",
                (pl.col("per_metric").mean() * 100).round(1),
                (pl.col("per_metric").min() * 100).round(1),
                (pl.col("per_metric").max() * 100).round(1),
            ).alias("metrics")
        )
        .pivot(
            on="model",
            index=metric,
            values="metrics",
        )
        .sort(metric)
    )
metric = "region_str"

with pl.Config(set_tbl_cols=500, set_tbl_rows=500):
    print(group(df, metric))

print(
    df
    .group_by(["model", metric])
    .agg(
        pl.len().alias("rows"),
        pl.col("result").sum().alias("correct"),
        pl.col("result").mean().alias("accuracy"),
        pl.col("raw_model_answer").n_unique().alias("answers"),
    )
    .sort(["model", metric])
)

shape: (43, 9)
┌──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│ region_s ┆ gpt-5-20 ┆ gemini-3 ┆ gemini-3 ┆ gpt-3.5- ┆ gpt-3.5- ┆ gpt-4-06 ┆ gemini-2 ┆ gpt-5.5- │
│ tr       ┆ 25-08-07 ┆ .5-flash ┆ .1-flash ┆ turbo-11 ┆ turbo-01 ┆ 13       ┆ .5-flash ┆ 2026-04- │
│ ---      ┆ ---      ┆ ---      ┆ -lite    ┆ 06       ┆ 25       ┆ ---      ┆ ---      ┆ 23       │
│ str      ┆ str      ┆ str      ┆ ---      ┆ ---      ┆ ---      ┆ str      ┆ str      ┆ ---      │
│          ┆          ┆          ┆ str      ┆ str      ┆ str      ┆          ┆          ┆ str      │
╞══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╡
│ null     ┆ null     ┆ 0.0      ┆ 0.0      ┆ null     ┆ null     ┆ 0.0      ┆ 0.0      ┆ 0.0      │
│          ┆          ┆ [0.0,    ┆ [0.0,    ┆          ┆          ┆ [0.0,    ┆ [0.0,    ┆ [0.0,    │
│          ┆          ┆ 0.0]     ┆ 0.0]     ┆          ┆          ┆ 0.0]    

In [4]:
print(df.columns)
print(df["macro_str"].unique())

with pl.Config(set_tbl_rows=500):
    print(
        df
        .select(["model", "endpoint_identifier", "raw_model_answer", metric])
        .group_by(["model", "endpoint_identifier", metric, "raw_model_answer"])
        .agg(pl.len().alias("per_metric"))
    )

['endpoint_identifier', 'entry_idx', 'model', 'db_model', 'reasoning', 'raw_model_answer', 'raw_ground_truth', 'mapped_model_answer', 'result', 'message', 'region_idx', 'region_str', 'macro_idx', 'macro_str', 'section_idx', 'section_str', 'subsection_idx', 'subsection_str', 'year_range']
shape: (11,)
Series: 'macro_str' [str]
[
	null
	"Southeast Asia"
	"Europe"
	"North America"
	"Central and Northern Eurasia"
	…
	"Africa"
	"Southwest Asia"
	"East Asia"
	"South Asia"
	"Oceania-Australia"
]
shape: (38_460, 5)
┌──────────────────────┬─────────────────────┬─────────────────────┬──────────────────┬────────────┐
│ model                ┆ endpoint_identifier ┆ region_str          ┆ raw_model_answer ┆ per_metric │
│ ---                  ┆ ---                 ┆ ---                 ┆ ---              ┆ ---        │
│ str                  ┆ str                 ┆ str                 ┆ str              ┆ u32        │
╞══════════════════════╪═════════════════════╪═════════════════════╪═══════════════

In [5]:
# Filter down to Qwen-7B-Chat where the endpoint score is exactly 0
zero_endpoints = (
    df
    .filter(pl.col("llm_model") == "Qwen-7B-Chat")
    .group_by(["endpoint", metric])
    .agg(pl.col("result").mean().alias("per_metric"))
    .filter(pl.col("per_metric") == 0)
)

print(zero_endpoints)

ColumnNotFoundError: unable to find column "llm_model"; valid columns: ["endpoint_identifier", "entry_idx", "model", "db_model", "reasoning", "raw_model_answer", "raw_ground_truth", "mapped_model_answer", "result", "message", "region_idx", "region_str", "macro_idx", "macro_str", "section_idx", "section_str", "subsection_idx", "subsection_str", "year_range"]

In [ ]:
accuracy_table = (
    df
    .group_by(["endpoint_identifier", "model"])
    .agg(
        (pl.col("result").mean() * 100)
        .round(2)
        .alias("accuracy_pct")
    )
    .pivot(
        on="model",
        index="endpoint_identifier",
        values="accuracy_pct",
    )
    .sort("endpoint_identifier")
)

with pl.Config(set_tbl_rows=-1, set_tbl_cols=-1):
    print(accuracy_table)

ColumnNotFoundError: unable to find column "result"; valid columns: ["endpoint_identifier", "entry_idx", "model", "db_model", "reasoning", "answer", "message", "region_idx", "region_str", "macro_idx", "macro_str", "section_idx", "section_str", "subsection_idx", "subsection_str", "year_range"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
DF ["endpoint_identifier", "entry_idx", "model", "db_model", ...]; PROJECT */16 COLUMNS